# Verify latest extraction results
Quick check of the latest rows in `availability_genai` and `files_genai_processed`.

In [ ]:
dbutils.widgets.text("catalog", "ltsa_dev")
dbutils.widgets.text("schema", "silver")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

In [ ]:
import json

output = {}

try:
    df_avail = spark.sql(f"""
        SELECT file, maintainer, country, type, turbine_id, date, value, reporting_date, run_date
        FROM {CATALOG}.{SCHEMA}.availability_genai
        ORDER BY run_date DESC
        LIMIT 20
    """)
    total = spark.table(f"{CATALOG}.{SCHEMA}.availability_genai").count()
    rows = [row.asDict() for row in df_avail.collect()]
    for r in rows:
        for k, v in r.items():
            if not isinstance(v, (str, int, float, bool, type(None))):
                r[k] = str(v)
    output["availability"] = rows
    output["total_rows"] = total
except Exception as e:
    output["availability_error"] = str(e)

try:
    df_tracking = spark.sql(f"""
        SELECT file, run_date, error
        FROM {CATALOG}.{SCHEMA}.files_genai_processed
        ORDER BY run_date DESC
        LIMIT 10
    """)
    tracking_rows = [row.asDict() for row in df_tracking.collect()]
    for r in tracking_rows:
        for k, v in r.items():
            if not isinstance(v, (str, int, float, bool, type(None))):
                r[k] = str(v)
    output["tracking"] = tracking_rows
except Exception as e:
    output["tracking_error"] = str(e)

dbutils.notebook.exit(json.dumps(output, indent=2, default=str))